In [60]:
import pandas as pd
import os
import json

with open("../config.json") as json_file:
    _LOCAL_CONFIG = json.load(json_file)

mskcc_csv_path = _LOCAL_CONFIG["mskcc_csv_path"]
mskcc_folder_path = _LOCAL_CONFIG["mskcc_folder_path"]
dataset_output_path = _LOCAL_CONFIG["output_folder_path"]

dt = pd.read_csv(mskcc_csv_path)
dt.head(2)

,isic_id,attribution,copyright_license,age_approx,anatom_site_general,anatom_site_special,concomitant_biopsy,dermoscopic_type,diagnosis_1,diagnosis_2,diagnosis_confirm_type,fitzpatrick_skin_type,image_type,lesion_id,melanocytic,patient_id,sex
0,ISIC_0077599,Memorial Sloan Kettering Cancer Center,CC-BY,65.0,anterior torso,NaN,False,contact non-polarized,Benign,Benign - Other,single contributor clinical assessment,IV,dermoscopic,IL_4740307,False,IP_5007819,male
1,ISIC_0082910,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,anterior torso,NaN,False,NaN,Benign,NaN,single contributor clinical assessment,V,clinical: close-up,IL_0206920,NaN,IP_2730930,female


In [61]:
dt['image_type'].unique()

array(['dermoscopic', 'clinical: close-up'], dtype=object)

In [62]:
df_2 = pd.read_csv(os.path.join(mskcc_folder_path, 'supplements/s2.csv'))
df_2.head()

,Unnamed: 0,tag_id,patient_id,lesion_id,type,anatomic_site
0,1,318918,IP_0131529,IL_4847840,lesion,upper back
1,2,319346,IP_0131529,IL_7743568,lesion,upper arm
2,3,589109,IP_0131529,IL_8056237,lesion,upper arm
3,4,604882,IP_0131529,IL_0716450,lesion,abdomen
4,5,676874,IP_0131529,IL_8792216,lesion,upper chest


In [63]:
dt = dt.merge(df_2, on='lesion_id', how='left')
dt['type'].unique()

array(['normal-skin', 'lesion'], dtype=object)

In [ ]:
df_normal = dt[dt['type'] == 'normal-skin']
df_normal = df_normal[['isic_id', 'lesion_id', 'patient_id_x', 'fitzpatrick_skin_type', 'type', 'age_approx', 'sex', 'anatom_site_general', 'image_type']]
df_normal.rename(columns={'patient_id_x': 'patient-id', 'isic_id' : 'img-id', 'lesion_id' : 'lesion-id', 'type' : 'clinicalDiagnostic', 'fitzpatrick_skin_type' : 'fitzpatrickSkinType', 'sex' : 'gender', 'age_approx' : 'age', 'anatom_site_general' : 'macroBodyRegion', 'image_type' : 'img-src'}, inplace=True)
df_normal['usePesticide'] = 'I'
df_normal['familySkinCancerHistory'] = 'I'
df_normal['familyCancerHistory'] = 'I'
df_normal['hasItched'] = 'False'
df_normal['hasGrown'] = 'False'
df_normal['hasHurt'] = 'False'
df_normal['hasChanged'] = 'False'
df_normal['hasBled'] = 'False'
df_normal['hasElevation'] = 'False'
df_normal['histoDiagnostic'] = 'EMPTY'
df_normal['histoMacroCID'] = 'EMPTY'
df_normal['clinicalMacroCID'] = '00'

In [65]:
ids_clinical = set(df_normal[df_normal['img-src'] == 'clinical: close-up']['lesion-id'])
ids_dermatoscope = set(df_normal[df_normal['img-src'] == 'dermoscopic']['lesion-id'])
ids_with_both = ids_clinical.intersection(ids_dermatoscope)

df_normal = df_normal[df_normal['lesion-id'].isin(ids_with_both)]
df_normal['img-src'].value_counts()

img-src
dermoscopic           1810
clinical: close-up     610
Name: count, dtype: int64

In [66]:
correction_dict = {
    'clinical: close-up': 'CLINICAL',
    'dermoscopic': 'DERMATOSCOPE'
}
df_normal['img-src'] = df_normal['img-src'].map(correction_dict)
df_normal['img-id'] = df_normal['img-id'].apply(lambda x: os.path.join(mskcc_folder_path ,'images', x + '.jpg'))

In [67]:
df_normal.to_csv(os.path.join(dataset_output_path, 'mskcc_normal_skin_cli_derm_metadata.csv'), index=False)